# G2 — SIMULATED DEMO · KHÔNG PHẢI KẾT QUẢ THỰC NGHIỆM

Notebook này chỉ minh họa bảng và biểu đồ cho báo cáo. Các giá trị do bộ sinh số ngẫu nhiên tạo ra, **không tính AU-PRO trên score map/ground truth**, không chạy model, không đọc checkpoint và không train.

**Không dùng dữ liệu này để tuyên bố r=128, d=512, Attention tốt hơn phương pháp khác.** Cấu hình đó là lựa chọn của người dùng, chưa được xác nhận. Mọi CSV/biểu đồ đều mang nhãn `SIMULATED`. Run All chạy trên CPU, không cần GPU/dataset.

E0 không có training loss; E4 và E5-average dùng chung một dòng minh họa. Không gộp E3 (3 Local sources) với E5-average (6 Local/Context sources).

In [ ]:
from pathlib import Path
import csv, json, math, random, zipfile
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import display, HTML
except ImportError:  # Colab có IPython; fallback cho chạy CPU bằng script.
    HTML = str
    def display(*values):
        for value in values:
            if isinstance(value, str): print(value)

LABEL = 'SIMULATED — KHÔNG PHẢI KẾT QUẢ THỰC NGHIỆM'
SEED, EPOCHS = 42, 20
OUTPUT = Path('DEMO_G2_SIMULATED')
OUTPUT.mkdir(parents=True, exist_ok=True)
CATEGORIES = ['can','fabric','fruit_jelly','rice','sheet_metal','vial','wallplugs','walnuts']
EXPERIMENTS = ['E0_memory_bank','E1','E2','E3','E4_E5_average',
               'E5_weighted','E5_concat','E5_gated','E5_attention']
FUSIONS = ['E4_E5_average','E5_weighted','E5_concat','E5_gated','E5_attention']
rng = random.Random(SEED)
display(HTML(f'<h2 style="color:#b71c1c">{LABEL}</h2><p>Actual training/evaluation runs: <b>0</b>.</p>'))

## 1. Sinh placeholders minh họa

Dùng cùng phân bố cho mọi phương pháp; không gán lợi thế cho Attention và không ép thứ hạng. “DEV AU-PRO” dưới đây là **placeholder ngẫu nhiên**, không phải metric đo được. Loss là đường cong nhân tạo có nhiễu, không phải loss của MS-ILA.

In [ ]:
history, metrics = [], []
for category in CATEGORIES:
    for experiment in EXPERIMENTS:
        base = dict(evidence_type='SIMULATED', category=category, experiment=experiment,
                    real_training_status='NOT RUN', real_evaluation_status='NOT RUN',
                    checkpoint=None, seed=SEED,
                    r=128 if experiment not in {'E0_memory_bank','E1'} else None,
                    d=512 if experiment not in {'E0_memory_bank','E1'} else None)
        if experiment == 'E0_memory_bank':
            metrics.append(dict(base, demo_best_epoch=None, demo_train_loss=None,
                                demo_dev_aupro_005_placeholder=rng.uniform(.25,.8)))
            continue
        initial = rng.uniform(.85,1.35)
        floor = rng.uniform(.12,.35)
        target = rng.uniform(.25,.8)
        run = []
        for epoch in range(1,EPOCHS+1):
            loss = max(.01, floor + initial*math.exp(-epoch/6) + rng.gauss(0,.018))
            score = min(1.,max(0.,target-.15*math.exp(-epoch/5)+rng.gauss(0,.025)))
            row = dict(base, epoch=epoch, demo_train_loss=loss,
                       demo_dev_aupro_005_placeholder=score)
            run.append(row); history.append(row)
        best = max(run,key=lambda row:row['demo_dev_aupro_005_placeholder'])
        metrics.append(dict(base, demo_best_epoch=best['epoch'],
                            demo_train_loss=best['demo_train_loss'],
                            demo_dev_aupro_005_placeholder=best['demo_dev_aupro_005_placeholder']))
assert len(metrics)==9*8 and len(history)==8*8*EPOCHS
assert all(row['evidence_type']=='SIMULATED' and row['checkpoint'] is None for row in metrics)
print(LABEL, '| Placeholder rows:',len(metrics),'| Actual runs: 0')

## 2. Bảng theo category và macro DEMO

Bảng này có đủ placeholders cho tám category, **coverage thực nghiệm vẫn 0/8**. Macro là trung bình của placeholders, không chứng minh cấu hình nào tốt hơn.

In [ ]:
columns=['Category',*EXPERIMENTS]
lookup={(r['category'],r['experiment']):r for r in metrics}
html=f'<h3>{LABEL}</h3><table border="1"><tr>'+''.join(f'<th>{c}</th>' for c in columns)+'</tr>'
for category in CATEGORIES:
    html += '<tr><td>'+category+'</td>'+''.join(
        f"<td>{lookup[category,e]['demo_dev_aupro_005_placeholder']:.4f}<br>SIMULATED</td>" for e in EXPERIMENTS)+'</tr>'
macro={e:sum(lookup[c,e]['demo_dev_aupro_005_placeholder'] for c in CATEGORIES)/8 for e in EXPERIMENTS}
html += '<tr><td>Macro DEMO</td>'+''.join(f'<td>{macro[e]:.4f}<br>SIMULATED</td>' for e in EXPERIMENTS)+'</tr></table>'
display(HTML(html))
print('Real coverage: 0/8 per experiment. Selected configuration: user choice, UNVALIDATED.')

## 3. Loss/DEV curves và ma trận 5 Fusion × 8 category — SIMULATED

Không diễn giải thứ hạng của các số ngẫu nhiên thành kết luận về MS-ILA. Các ảnh được gắn nhãn demo cả trong nội dung và tên file.

In [ ]:
fig,axes=plt.subplots(4,2,figsize=(13,13))
for category,ax in zip(CATEGORIES,axes.flat):
    for experiment in EXPERIMENTS[1:]:
        rows=[r for r in history if r['category']==category and r['experiment']==experiment]
        ax.plot([r['epoch'] for r in rows],[r['demo_train_loss'] for r in rows],label=experiment)
    ax.set_title(category+' — SIMULATED'); ax.set_xlabel('Demo epoch'); ax.set_ylabel('Artificial loss')
axes.flat[0].legend(fontsize=7)
fig.suptitle(LABEL,fontsize=13,color='darkred'); fig.tight_layout(rect=(0,0,1,.96))
fig.savefig(OUTPUT/'SIMULATED_loss_curves.png',dpi=140); display(fig); plt.close(fig)

fig,axes=plt.subplots(4,2,figsize=(13,13))
for category,ax in zip(CATEGORIES,axes.flat):
    for experiment in FUSIONS:
        rows=[r for r in history if r['category']==category and r['experiment']==experiment]
        ax.plot([r['epoch'] for r in rows],[r['demo_dev_aupro_005_placeholder'] for r in rows],label=experiment)
    ax.set_title(category+' — SIMULATED'); ax.set_xlabel('Demo epoch')
    ax.set_ylabel('DEV AU-PRO placeholder, NOT measured'); ax.set_ylim(0,1)
axes.flat[0].legend(fontsize=7)
fig.suptitle(LABEL,fontsize=13,color='darkred'); fig.tight_layout(rect=(0,0,1,.96))
fig.savefig(OUTPUT/'SIMULATED_dev_curves.png',dpi=140); display(fig); plt.close(fig)

matrix=np.array([[lookup[c,e]['demo_dev_aupro_005_placeholder'] for c in CATEGORIES] for e in FUSIONS])
fig,ax=plt.subplots(figsize=(12,4)); im=ax.imshow(matrix,vmin=0,vmax=1,cmap='viridis',aspect='auto')
ax.set_xticks(range(8),CATEGORIES,rotation=30,ha='right'); ax.set_yticks(range(5),FUSIONS)
for i in range(5):
    for j in range(8): ax.text(j,i,f'{matrix[i,j]:.3f}\nDEMO',ha='center',va='center',fontsize=8,color='white')
ax.set_title(LABEL+'\nRandom placeholders, not measured AU-PRO',color='darkred')
fig.colorbar(im,ax=ax,label='SIMULATED placeholder'); fig.tight_layout()
fig.savefig(OUTPUT/'SIMULATED_fusion_matrix.png',dpi=140); display(fig); plt.close(fig)

## 4. Export DEMO riêng biệt

Không tạo `metrics.json`, `best.pt`, selection lock hoặc file có tên trùng kết quả G2 thật. Trong mọi dòng CSV có `evidence_type=SIMULATED`, checkpoint trống và trạng thái thực `NOT RUN`. Giữ nguyên nhãn này khi dùng làm ví dụ bố cục.

Muốn bảo vệ lựa chọn r128/d512/Attention bằng bằng chứng thật: dùng `G2_05_E5_Fixed_Attention.ipynb`; để kết luận hơn phương pháp khác cần chạy đối chứng cùng protocol.

In [ ]:
def write_csv(name,rows):
    with (OUTPUT/name).open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]))
        writer.writeheader(); writer.writerows(rows)
write_csv('SIMULATED_history.csv',history)
write_csv('SIMULATED_metrics_placeholders.csv',metrics)
write_csv('SIMULATED_macro.csv',[dict(evidence_type='SIMULATED',experiment=e,
    demo_macro_placeholder=macro[e],real_training_status='NOT RUN',real_evaluation_status='NOT RUN') for e in EXPERIMENTS])
summary=dict(status='DEMO_ONLY',evidence_type='SIMULATED',real_training_runs=0,real_evaluation_runs=0,
    placeholder_rows=len(metrics),selection_status='user_choice_unvalidated',selection_lock=None,
    purpose='Layout demonstration; random placeholders cannot validate model accuracy or superiority.')
(OUTPUT/'SIMULATED_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
(OUTPUT/'README_SIMULATED.txt').write_text(LABEL+'\nAll numbers are random placeholders. Real training/evaluation: NOT RUN.\n')
archive=Path('DEMO_G2_SIMULATED.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT.iterdir()): bundle.write(path,arcname=str(path))
print(LABEL); print(summary); print('Demo export:',archive.resolve())
# Optional Colab download: from google.colab import files; files.download(str(archive))